# **RI Loan Default Demo**

> ▶️ **Try this in Colab!** Run the [RI Lending Classification Walkthrough in Google Colab](https://colab.research.google.com/github/RobustIntelligence/docs/blob/main/notebooks/demo_notebooks/RI_Lending_Classification_Walkthrough.ipynb). 

You are a data scientist at a Bank. The data science team has been tasked with implementing a binary classification model to predict whether an individual will default on a loan. The goal of this project is two-fold: we want to monitor how that model performs over time as well as ensure that the model is compliant with financial regulations. In order to accomplish the latter, we will be testing whether the model is biased against certain protected features. One could imagine such models being used downstream for various purposes, such as loan approval or funding allocation. A biased model could yield disadvantageous outcomes for protected groups. For instance, we may find that an individual with a specific race or race/gender combination causes the model to consistently predict a higher probability of them defaulting, causing a higher rate of loan rejection.
    

In this Notebook Walkthrough, we will walkthrough our core products of **AI Stress Testing** and **AI Continuous Testing** in a *Bias and Fairness* setting. RIME AI Stress Testing allows you to test the developed model and datasets. With this compliance-focused setting, you will be able to verify your AI model for bias and fairness issues. RIME AI Continuous Testing allows you to continue monitoring your deployed model for bias.

## 1. **Install Dependencies, Import Libraries and Download Data**

In [ ]:
# Installing the dependencies
%pip install rime-sdk
%pip install python-dotenv
%pip install https://github.com/RobustIntelligence/ri-public-examples/archive/master.zip    

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from rime_sdk import Client
import os
from dotenv import load_dotenv, find_dotenv
from ri_public_examples.download_files import download_files

In [ ]:
download_files('tabular-2.0/lending', 'lending')

## 2. **Establish the RI Client**

To get started, provide the API credentials and the base domain/address of the RIME service. You can generate and copy an API token from the API Access Tokens Page under Workspace settings. For the domian/address of the RIME service, contact your admin. 

![img_1](https://drive.google.com/uc?id=1t5GG5UoYljABIqCKS4FucLmMzEuIkZAP)
![img_2](https://drive.google.com/uc?id=1rdJQBOwY4lj74Zco5q6yO-NXvV5aj2GN)

In [ ]:
#Load environment variables
load_dotenv(find_dotenv())
API_TOKEN = os.environ.get('API_TOKEN')
CLUSTER_URL = os.environ.get('CLUSTER_URL')
AGENT_ID = os.environ.get('AGENT_ID')
WORKSPACE_ID = os.environ.get('WORKSPACE_ID')

In [ ]:
client = Client(CLUSTER_URL, API_TOKEN)

## 3. **Create a New Project**

You can create projects in RIME to organize your test runs. Each project represents a workspace for a given machine learning task. It can contain multiple candidate models, but should only contain one promoted production model.  

In [ ]:
description = (
    "Run Stress Testing on a tabular"
    " binary classification model and dataset."
    " Demonstration uses the Lending Club dataset, which is used"
    " to predict whether someone will repay a loan."
)
project = client.create_project(
    name="Loan Default Engine",
    description=description,
    model_task="MODEL_TASK_BINARY_CLASSIFICATION"
)


## 4. **Uploading the Model and Datasets**

##### 4.1. Uploading Loan Default Model

In [ ]:
model_dir = client.upload_directory(
    Path('lending/models'), 
    upload_path = "ri_public_examples_lending"
)

model_path = model_dir + "/model.py"

model_id = project.register_model_from_path(
    name = f"model_{datetime.now()}", 
    remote_path = model_path,
    agent_id = AGENT_ID)

##### 4.2.a Uploading and registering the reference dataset (training data)

In [ ]:
ref_s3_path = client.upload_file(
    Path('lending/data/ref.csv'), 
    upload_path = "ri_public_examples_lending"
)

ref_dataset_id = project.register_dataset_from_file(
    name = f"ref_dataset_{datetime.now()}",
    remote_path = ref_s3_path,
    data_params = {
        "label_col": "loan_status",
        "protected_features": [
            "sex", 
            "race", 
            "addr_state"
        ],
        "sample": False
    },
    agent_id = AGENT_ID
)

##### 4.2.b Uploading and registering reference labels dataset 

In [ ]:
ref_preds_s3_path = client.upload_file(
    Path("lending/data/ref_preds.csv"),
    upload_path = "ri_public_examples_lending"
)

ref_pred_id = project.register_predictions_from_file(
    dataset_id = ref_dataset_id, 
    model_id = model_id, 
    remote_path = ref_preds_s3_path, 
    agent_id = AGENT_ID)

##### 4.3.a Uploading and registering evaluation dataset 

In [ ]:
eval_s3_path = client.upload_file(
    Path('lending/data/eval.csv'), 
    upload_path = "ri_public_examples_lending"
)

eval_dataset_id = project.register_dataset_from_file(
    name = f"eval_dataset_{datetime.now()}",
    remote_path = eval_s3_path,
    data_params = {
        "label_col": "loan_status",
        "protected_features": [
            "sex", 
            "race", 
            "addr_state"
        ],
        "nrows": 100000,
        "sample": False
    },
    agent_id=AGENT_ID
)

##### 4.3.b Uploading and registering evaluation predictions dataset 

In [ ]:
eval_preds_s3_path = client.upload_file(
    Path("lending/data/eval_preds.csv"), 
    upload_path = "ri_public_examples_lending"
)

eval_pred_id = project.register_predictions_from_file(
    dataset_id = eval_dataset_id, 
    model_id = model_id, 
    remote_path = eval_preds_s3_path, 
    agent_id = AGENT_ID)

## 5. **Running a Stress Test**

In [ ]:
stress_test_config = {
    "data_info": {
        "ref_dataset_id": ref_dataset_id,
        "eval_dataset_id": eval_dataset_id
    },
    "model_id": model_id,
    "run_name": "Loan Default Prediction",
    "categories": [
        "TEST_CATEGORY_TYPE_ABNORMAL_INPUTS",
        "TEST_CATEGORY_TYPE_ADVERSARIAL",
        "TEST_CATEGORY_TYPE_BIAS_AND_FAIRNESS",
        "TEST_CATEGORY_TYPE_DATA_CLEANLINESS",
        "TEST_CATEGORY_TYPE_DRIFT",
        "TEST_CATEGORY_TYPE_DATA_POISONING_DETECTION",
        "TEST_CATEGORY_TYPE_MODEL_PERFORMANCE",
        "TEST_CATEGORY_TYPE_SUBSET_PERFORMANCE",
        "TEST_CATEGORY_TYPE_SUBSET_PERFORMANCE_DEGRADATION",
        "TEST_CATEGORY_TYPE_TRANSFORMATIONS",
    ],
    "run_time_info": {
        "random_seed" : "0"
    }
}
stress_job = client.start_stress_test(
    test_run_config = stress_test_config, 
    project_id = project.project_id, 
    agent_id = AGENT_ID
)
stress_job.get_status(verbose = True, wait_until_finish = True)

## 6. **Analyzing and Querying Results**

In [ ]:
test_run = stress_job.get_test_run()
results_df = test_run.get_result_df()
results_df.head()

In [ ]:
# Get a link to the stress test
print("https://"+ test_run.get_link())